In [2]:
import pandas as pd
import os

In [3]:
truthfulqa_train = pd.read_parquet(
    "../data/processed/truthfulqa_train.parquet"
)

halueval_train = pd.read_parquet(
    "../data/processed/halueval_train.parquet"
)

fever_train = pd.read_parquet(
    "../data/processed/fever_train.parquet"
)

print("TruthfulQA:", truthfulqa_train.shape)
print("HaluEval:", halueval_train.shape)
print("FEVER:", fever_train.shape)

TruthfulQA: (4166, 8)
HaluEval: (14000, 8)
FEVER: (9160, 8)


In [4]:
print("TruthfulQA columns:")
print(truthfulqa_train.columns.tolist())

print("\nHaluEval columns:")
print(halueval_train.columns.tolist())

print("\nFEVER columns:")
print(fever_train.columns.tolist())

TruthfulQA columns:
['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category']

HaluEval columns:
['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category']

FEVER columns:
['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category']


In [5]:
truthfulqa_train["split"] = "train"
halueval_train["split"] = "train"
fever_train["split"] = "train"

In [6]:
train_unified = pd.concat(
    [
        truthfulqa_train,
        halueval_train,
        fever_train
    ],
    ignore_index=True
)

In [7]:
print("Unified training shape:", train_unified.shape)

Unified training shape: (27326, 9)


In [8]:
print(
    train_unified["source_dataset"].value_counts()
)

source_dataset
halueval      14000
fever          9160
truthfulqa     4166
Name: count, dtype: int64


In [9]:
print(
    train_unified
    .groupby("source_dataset")["label"]
    .value_counts()
)

source_dataset  label
fever           1        4591
                0        4569
halueval        0        7000
                1        7000
truthfulqa      1        2354
                0        1812
Name: count, dtype: int64


In [10]:
print(train_unified.shape)

(27326, 9)


In [11]:
print(train_unified.columns.tolist())

['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category', 'split']


In [12]:
train_unified["has_context"] = (
    train_unified["context"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .astype(int)
)

In [13]:
train_unified["question_length"] = (
    train_unified["question"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

train_unified["answer_length"] = (
    train_unified["answer"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

train_unified["context_length"] = (
    train_unified["context"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

In [14]:
print(
    train_unified[
        [
            "question_length",
            "answer_length",
            "context_length"
        ]
    ].describe()
)

       question_length  answer_length  context_length
count     27326.000000   27326.000000    27326.000000
mean         13.672876       7.442106       28.447047
std           8.839156       5.204171       31.822223
min           2.000000       1.000000        0.000000
25%           8.000000       3.000000        0.000000
50%          11.000000       7.000000       24.000000
75%          16.000000      10.000000       52.000000
max         100.000000      61.000000      256.000000


In [15]:
print("Shape:", train_unified.shape)
print("Columns:", train_unified.columns.tolist())

Shape: (27326, 13)
Columns: ['question_id', 'candidate_id', 'source_dataset', 'question', 'answer', 'context', 'label', 'question_category', 'split', 'has_context', 'question_length', 'answer_length', 'context_length']


In [16]:
required_columns = [
    "question_id",
    "candidate_id",
    "source_dataset",
    "question",
    "answer",
    "context",
    "label",
    "question_category",
    "split"
]

missing_columns = [
    col
    for col in required_columns
    if col not in train_unified.columns
]

print("Missing columns:", missing_columns)

Missing columns: []


In [17]:
print(
    train_unified.dtypes
)

question_id            str
candidate_id           str
source_dataset         str
question               str
answer                 str
context                str
label                int64
question_category      str
split                  str
has_context          int64
question_length      int64
answer_length        int64
context_length       int64
dtype: object


In [18]:
train_unified["label"] = (
    train_unified["label"]
    .astype(int)
)

In [19]:
for column in [
    "question",
    "answer",
    "context"
]:
    
    empty_count = (
        train_unified[column]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
    
    print(
        column,
        "empty:",
        empty_count
    )

question empty: 0
answer empty: 0
context empty: 13326


In [20]:
train_unified["has_context"] = (
    train_unified["context"]
    .fillna("")
    .str.strip()
    .ne("")
    .astype(int)
)

In [21]:
print(
    train_unified["has_context"].value_counts()
)

has_context
1    14000
0    13326
Name: count, dtype: int64


In [22]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [23]:
from src.features.basic_features import add_basic_features

In [24]:
train_unified = add_basic_features(train_unified)

In [25]:
print(train_unified.shape)

(27326, 13)


In [26]:
truthfulqa_val = pd.read_parquet(
    "../data/processed/truthfulqa_validation.parquet"
)

halueval_val = pd.read_parquet(
    "../data/processed/halueval_validation.parquet"
)

fever_val = pd.read_parquet(
    "../data/processed/fever_validation.parquet"
)

In [27]:
truthfulqa_val["split"] = "validation"
halueval_val["split"] = "validation"
fever_val["split"] = "validation"

In [28]:
validation_unified = pd.concat(
    [
        truthfulqa_val,
        halueval_val,
        fever_val
    ],
    ignore_index=True
)

In [29]:
validation_unified = add_basic_features(
    validation_unified
)

In [30]:
truthfulqa_test = pd.read_parquet(
    "../data/processed/truthfulqa_test.parquet"
)

halueval_test = pd.read_parquet(
    "../data/processed/halueval_test.parquet"
)

fever_test = pd.read_parquet(
    "../data/processed/fever_test.parquet"
)

In [31]:
truthfulqa_test["split"] = "test"
halueval_test["split"] = "test"
fever_test["split"] = "test"

In [32]:
test_unified = pd.concat(
    [
        truthfulqa_test,
        halueval_test,
        fever_test
    ],
    ignore_index=True
)

In [33]:
test_unified = add_basic_features(
    test_unified
)

In [34]:
print("Train:", train_unified.shape)
print("Validation:", validation_unified.shape)
print("Test:", test_unified.shape)

Train: (27326, 13)
Validation: (5839, 13)
Test: (5801, 13)


In [35]:
print(
    train_unified[
        [
            "has_context",
            "question_length",
            "answer_length",
            "context_length"
        ]
    ].head()
)

   has_context  question_length  answer_length  context_length
0            0                9              2               0
1            0                9              4               0
2            0                9              8               0
3            0                9              7               0
4            0                9              6               0


In [36]:
train_unified.to_parquet(
    "../data/processed/unified_train_basic_features.parquet",
    index=False
)

validation_unified.to_parquet(
    "../data/processed/unified_validation_basic_features.parquet",
    index=False
)

test_unified.to_parquet(
    "../data/processed/unified_test_basic_features.parquet",
    index=False
)

print("Saved all three feature-ready datasets.")

Saved all three feature-ready datasets.


In [37]:
import os

for filename in [
    "unified_train_basic_features.parquet",
    "unified_validation_basic_features.parquet",
    "unified_test_basic_features.parquet"
]:
    path = f"../data/processed/{filename}"
    print(filename, "->", os.path.exists(path))

unified_train_basic_features.parquet -> True
unified_validation_basic_features.parquet -> True
unified_test_basic_features.parquet -> True


In [38]:
%pip install spacy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
import sys
print(sys.executable)
print(sys.version)

/opt/homebrew/opt/python@3.11/bin/python3.11
3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 21.0.0 (clang-2100.0.123.102)]


In [40]:
%pip install spacy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [41]:
!{sys.executable} -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 121.8 kB/s  0:02:190:00:0200:04

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [42]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("spaCy:", spacy.__version__)
print("Model loaded successfully!")

spaCy: 3.8.15
Model loaded successfully!


In [43]:
train_unified.to_parquet(
    "../data/processed/unified_train_entity_features.parquet",
    index=False
)

validation_unified.to_parquet(
    "../data/processed/unified_validation_entity_features.parquet",
    index=False
)

test_unified.to_parquet(
    "../data/processed/unified_test_entity_features.parquet",
    index=False
)

In [44]:
import os

for filename in [
    "unified_train_entity_features.parquet",
    "unified_validation_entity_features.parquet",
    "unified_test_entity_features.parquet"
]:
    path = f"../data/processed/{filename}"
    print(filename, "->", os.path.exists(path))

unified_train_entity_features.parquet -> True
unified_validation_entity_features.parquet -> True
unified_test_entity_features.parquet -> True
